In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA L40


In [2]:
import unsloth
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only
import json
from transformers import EarlyStoppingCallback
import random

import pandas as pd
from tqdm import tqdm
from peft import PeftModel
import sys

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/shivraj-pg/miniconda3/envs/stableenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/home/shivraj-pg/poetrygen_new/Trained_Models/Phi4-14B-DEV-0SHOT/checkpoint-1000",
    max_seq_length=1300,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.7.3: Fast Llama patching. Transformers: 5.14.1.
   \\   /|    NVIDIA L40. Num GPUs = 1. Max memory: 44.392 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 363/363 [00:02<00:00, 170.03it/s]
Unsloth 2026.7.3 patched 40 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
ds = load_dataset("csv", data_files="/home/shivraj-pg/poetrygen_new/Files/v3_gitapress_final.csv")["train"]

train_ds = ds.filter(lambda x: x["split"] == "train")

print("Train:", len(train_ds))
print(train_ds.column_names)

Train: 23346
['sa', 'hi', 'skr_out', 'syllable_count', 'skr_meter', 'meter_cd', 'comments', 'prompt', 'split']


In [6]:
def format_grpo_example(example):
    return {
        "prompt": [
            {
                "role": "system",
                "content": example["prompt"],
            },
            {
                "role": "user",
                "content": (
                    f"Meaning:\n{example['hi']}\n\n"
                    "Generate the Sanskrit verse.\n"
                ),
            },
        ],
        "meter_cd": example["meter_cd"],
        "syllable_count": int(example["syllable_count"]),
    }


train_ds = train_ds.map(
    format_grpo_example,
    remove_columns=train_ds.column_names,
)

print(train_ds[0])

Map: 100%|██████████| 23346/23346 [00:02<00:00, 9900.72 examples/s] 

{'syllable_count': 56, 'meter_cd': 'Vasantatilakā', 'prompt': [{'role': 'system', 'content': 'Objective:\n\nGenerate a sanskrit verse in Vasantatilakā meter based on a given meaning or theme while strictly following the metrical (?and rhyme) rules.\n\nRules of Chandas:\nSyllable Rules:\nLAGHU vowels: अ, इ, उ, ऋ, ऌ\nGURU vowels: आ, ई, ऊ, ॠ, ॡ, ए, ऐ, ओ, औ\nHRASVA vowels: अ, इ, उ, ऋ, ऌ\nDEERGHA vowels: आ, ई, ऊ, ॠ, ॡ, ए, ऐ, ओ, औ\nSyllable is marked laghu, guru and hrasva, deergha based on the vowel it contains.\nSyllable containing anusvāra("ं") or visarga("ः") is always marked as guru.\nSyllable that is followed by a conjunct consonant (saṁyuktākṣara)\nis always marked guru.\n\nVerse Rules:\nThe verse contains 56 syllables (akṣaras) and 4 pādas (lines) in total, with each pāda containing 14 syllables.\nThe pattern of syllables in every pāda must be: तगण (GGL), भगण (GLL), जगण (LGL), जगण (LGL), followed by two Guru (GG) syllables.\nThe complete pattern for each pāda is: GGL GLL LGL LGL GG.\

### LG Pattern Extractor

In [8]:
from skrutable.meter_identification import MeterIdentifier
import re
MI = MeterIdentifier()


import re

def extract_lg_pattern(verse, verbose=False):
    if verbose: print(verse)
    verse = MI.identify_meter(
            verse,
            from_scheme='DEV',
            resplit_option='resplit_lite'
        )
    
    summary = verse.summarize()
    if verbose: print(summary)

    patterns = re.findall(
        r'^([lg]+)\s+\{m:',
        summary,
        re.MULTILINE
    )[:4]

    return ''.join(patterns)

In [9]:
def add_target_lg(example):
    return {
        "target_lg_pattern": extract_lg_pattern(example["sa"])
    }

ds = ds.map(
    add_target_lg,
    desc="Extracting target LG patterns"
)

Extracting target LG patterns: 100%|██████████| 29183/29183 [00:16<00:00, 1755.05 examples/s]


In [10]:
print(
    ds.select_columns(
        ["meter_cd", "syllable_count", "target_lg_pattern"]
    ).select(range(10))
)

Dataset({
    features: ['meter_cd', 'syllable_count', 'target_lg_pattern'],
    num_rows: 10
})


In [12]:
ds = ds.select([i for i in range(len(ds)) if i != 1138]) # 1138 has a foul examples
for row in ds:
    assert len(row["target_lg_pattern"]) == int(row["syllable_count"]), (
        f"Mismatch: {row['meter_cd']} | "
        f"expected={row['syllable_count']} | "
        f"got={len(row['target_lg_pattern'])}"
    )

print("All LG patterns match syllable counts.")

All LG patterns match syllable counts.


# Reward Function

In [ ]:
import re

def meter_reward_single(
    generated_text,
    target_lg_pattern,
    target_syllable_count,
):
    # ----------------------------------
    # 1. Reject English / numbers
    # ----------------------------------
    if re.search(r"[A-Za-z0-46-9]", generated_text):
        return 0.0

    # ----------------------------------
    # 2. Extract generated L/G pattern
    # ----------------------------------
    try:
        generated_lg = extract_lg_pattern(generated_text)
    except Exception:
        return 0.0

    if not generated_lg:
        return 0.0

    # ----------------------------------
    # 3. Syllable count
    # ----------------------------------
    if len(generated_lg) != int(target_syllable_count):
        return 0.0

    # ----------------------------------
    # 4. Safety check
    # ----------------------------------
    if len(generated_lg) != len(target_lg_pattern):
        return 0.0

    # ----------------------------------
    # 5. Position-wise L/G match
    # ----------------------------------
    matches = sum(
        generated == target
        for generated, target
        in zip(generated_lg, target_lg_pattern)
    )

    return matches / len(target_lg_pattern)